# Lab 1: Tokenização, Embeddings, Logits, Temperatura

## Lab 1: Tokenização, Embeddings, Logits e Temperatura

Usamos `HuggingFaceTB/SmolLM2-135M` — um modelo pequeno (135M parâmetros)
mas **realmente treinado** (não uma arquitetura aleatória), pra ver
comportamento de linguagem genuíno, não só mecânica vazia.

In [1]:
!pip install -q torch transformers

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()
print(f"✓ Modelo carregado: {sum(p.numel() for p in model.parameters()):,} parâmetros")

✓ Modelo carregado: 134,515,008 parâmetros


### 1. Tokenização — texto vira números

In [2]:
text = "AI Engineering combina LLMs, dados reais e engenharia de software."
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print(f"Texto: {text}")
print(f"Tokens ({len(tokens)}): {tokens}")
print(f"IDs: {token_ids}")

Texto: AI Engineering combina LLMs, dados reais e engenharia de software.
Tokens (18): ['AI', 'ĠEngineering', 'Ġcomb', 'ina', 'ĠLL', 'Ms', ',', 'Ġd', 'ados', 'Ġre', 'ais', 'Ġe', 'Ġeng', 'enh', 'aria', 'Ġde', 'Ġsoftware', '.']
IDs: [13701, 8574, 1775, 1787, 36669, 16653, 28, 287, 18514, 298, 10117, 297, 1227, 15413, 6702, 367, 3197, 30]


**Resultado esperado:** o texto vira uma lista de ~15-20 tokens — repare
que "Engineering" pode virar 1 token só, enquanto palavras em português
como "combina"/"engenharia" tendem a fragmentar em mais sub-tokens (o
vocabulário do modelo foi treinado majoritariamente em inglês).

### 2. Embeddings — cada token é um vetor

In [3]:
embedding_layer = model.get_input_embeddings()
input_ids = torch.tensor([token_ids])
embeddings = embedding_layer(input_ids)

print(f"Shape dos embeddings: {embeddings.shape}")  # (1, n_tokens, embedding_dim)
print(f"Vetor do primeiro token (5 primeiras dims): {embeddings[0][0][:5]}")

Shape dos embeddings: torch.Size([1, 18, 576])
Vetor do primeiro token (5 primeiras dims): tensor([-0.0251, -0.0075,  0.0209, -0.0173,  0.0103], dtype=torch.bfloat16,
       grad_fn=<SliceBackward0>)


**Resultado esperado:** shape `(1, N, 576)` — cada um dos N tokens virou um
vetor de 576 números (a dimensão de embedding do SmolLM2-135M).

### 3. Logits e Softmax — como o modelo "vota" no próximo token

In [4]:
import torch.nn.functional as F

prompt = "A capital da França é"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits[0, -1]  # logits pro PRÓXIMO token, após o último da sequência

probs = F.softmax(logits, dim=-1)
top5 = torch.topk(probs, 5)

print(f"Prompt: '{prompt}'")
print("Top 5 próximos tokens mais prováveis:")
for prob, idx in zip(top5.values, top5.indices):
    print(f"  '{tokenizer.decode([idx])}' — {prob.item():.1%}")

Prompt: 'A capital da França é'
Top 5 próximos tokens mais prováveis:
  ' um' — 16.7%
  ' u' — 12.2%
  ' o' — 6.1%
  ' a' — 4.8%
  ' un' — 1.6%


**Resultado esperado:** os logits (números "crus") viram probabilidades
reais somando 100% via Softmax — e o token mais provável deve fazer
sentido gramaticalmente como continuação de "A capital da França é" (mesmo
que o conteúdo factual de um modelo tão pequeno nem sempre esteja certo).

### 4. Temperatura, Top-K e Top-P — controlando a aleatoriedade

In [5]:
def sample_with_settings(prompt, temperature=1.0, top_k=50, top_p=1.0, n=3):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = []
    for _ in range(n):
        out = model.generate(
            **inputs, max_new_tokens=15, do_sample=True,
            temperature=temperature, top_k=top_k, top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )
        outputs.append(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    return outputs

prompt = "The best way to learn programming is"

print("🌡️  Temperature=0.1 (quase determinístico):")
for o in sample_with_settings(prompt, temperature=0.1):
    print(f"  {o!r}")

print("\n🌡️  Temperature=1.5 (mais aleatório):")
for o in sample_with_settings(prompt, temperature=1.5):
    print(f"  {o!r}")

🌡️  Temperature=0.1 (quase determinístico):
  ' to learn it in a fun way.\n\nThe best way to learn'
  ' to learn it in a fun way.\n\nThe best way to learn'
  ' to learn it from scratch.\n\nThe best way to learn programming is'

🌡️  Temperature=1.5 (mais aleatório):
  ' the following steps:\n\n  1. Use a debugger - (most IDE'
  ' through lots of repetition and practice with some basic examples. You can learn at'
  ' to write simple programs for free and see what others have done there before the'


**Resultado esperado:** com temperature=0.1, as 3 gerações devem ser
idênticas ou quase idênticas entre si (pouca variação); com temperature=1.5,
as 3 devem divergir bastante — algumas podem até virar texto sem muito
sentido (temperatura alta demais sacrifica coerência por variedade).

**Próximos passos:** Semana 2 abre o que acontece *dentro* do bloco
Transformer que gerou esses logits — attention, QKV, multi-head.